# LoRA 手撕实现

## 1. 思想
全量微调大模型要更新全部权重 $W_0$，显存与计算大。LoRA 假设权重更新是低秩的：
$$W=W_0+\Delta W,\quad \Delta W=B A\ (B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times d},\ r\ll d)$$
冻结 $W_0$，只训 $A,B$，可训练参数从 $d^2$ 降到 $2rd$。

## 2. 初始化
- $A$ 用 Kaiming 初始化，$B$ 初始化为 **0**，保证训练初始 $\Delta W=BA=0$，模型从预训练状态起步，不破坏已学知识。
- 前向：$y=W_0 x + B(A x)$，可把 $BA$ 合并回 $W_0$ 做无额外开销推理。

## 3. 为何省显存
- 优化器状态（Adam 的 m/v）只对 $A,B$ 维护，不存 $W_0$ 的；$W_0$ 冻结无梯度。
- 多 adapter 推理时共享 $W_0$，只切 $A/B$，见 `Easy_AIInfra/19`。

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    def __init__(self, in_dim, out_dim, rank=4, alpha=8):
        super().__init__()
        self.base = nn.Linear(in_dim, out_dim, bias=False)
        for p in self.base.parameters():
            p.requires_grad_(False)                       # 冻结预训练权重
        self.A = nn.Parameter(torch.randn(in_dim, rank))
        self.B = nn.Parameter(torch.zeros(rank, out_dim)) # B=0 保证初始 ΔW=0
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scaling = alpha / rank                        # LoRA 缩放

    def forward(self, x):
        return self.base(x) + (x @ self.A @ self.B) * self.scaling

    def merge(self):
        """把 BA 合并进 base 权重，推理无额外开销"""
        with torch.no_grad():
            self.base.weight.add_(self.scaling * (self.A @ self.B))

In [ ]:
# 验证：参数量、初始 ΔW=0、合并前后等价
torch.manual_seed(0)
in_dim, out_dim, rank = 256, 256, 4
lora = LoRALinear(in_dim, out_dim, rank)

# 可训练参数量
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
full = in_dim * out_dim
print(f'可训练 {trainable} (={2*rank*in_dim}) vs 全量 {full}, 比例 {trainable/full:.3f}')

# 初始 ΔW=0 -> 输出等于 base
x = torch.randn(8, in_dim)
out0 = lora(x)
base_out = lora.base(x)
print('初始输出 == base:', torch.allclose(out0, base_out, atol=1e-6))

# 训练一步后合并等价
lora.A.data += 0.1; lora.B.data += 0.1
out_before = lora(x)
lora.merge()
out_after = lora.base(x)
print('合并前后等价:', torch.allclose(out_before, out_after, atol=1e-5))

## 小结 / 易错点
- 原仓库 `Lora.ipynb` 用 `math.sqrt(5)` 但未 `import math`，会 NameError，本版已补。
- **B 必须初始化为 0**（不是 A），否则初始 ΔW≠0 会破坏预训练。
- `scaling = alpha/rank` 用于把不同 rank 的更新量归到相近尺度，alpha 常取 2r。
- 推理可 `merge()` 消除额外计算；多 adapter 场景则不合并、按请求路由 A/B。
- LoRA 一般加在 Q/V 投影上效果最佳，全层加收益递减。

## ✅ 测试验证

In [ ]:
# 验证 LoRA 性质
import torch
import torch.nn as nn

# LoRA: y = Wx + BAx, 其中 B 是 d×r, A 是 r×d, r << d
# 关键性质:
# 1. 初始时 BA = 0（A 或 B 初始化为 0），不改变原模型输出
# 2. 参数量 = 2*d*r << d*d

d, r = 256, 8
# 原始线性层
W = nn.Linear(d, d, bias=False)
# LoRA 分解
A = nn.Linear(d, r, bias=False)  # r×d
B = nn.Linear(r, d, bias=False)  # d×r

# 性质1: B 初始化为 0 → 初始 LoRA 输出为 0
nn.init.zeros_(B.weight)
x = torch.randn(4, d)
lora_out = B(A(x))
assert torch.allclose(lora_out, torch.zeros_like(lora_out), atol=1e-6), \
    "LoRA 初始输出应为 0"
print("  ✓ LoRA 初始输出为 0（B=0 初始化）")

# 性质2: 参数量对比
original_params = d * d
lora_params = d * r + r * d  # A + B
ratio = lora_params / original_params
print(f"  ✓ 参数量: 原始={original_params}, LoRA={lora_params}, 比例={ratio:.2%}")
assert lora_params < original_params, "LoRA 参数应更少"

# 性质3: 训练后 LoRA 改变输出
nn.init.kaiming_uniform_(A.weight)
nn.init.kaiming_uniform_(B.weight)
lora_out_trained = B(A(x))
assert not torch.allclose(lora_out_trained, torch.zeros_like(lora_out_trained), atol=1e-3), \
    "训练后 LoRA 输出应非零"
print("  ✓ 训练后 LoRA 输出非零")

print("✅ LoRA 测试通过: 零初始化、参数量减少、训练后有效")
